# Forward Kinematics

Source orientation: printed pages 137-170, PDF pages 155-188. This notebook is an original visualization-first lesson inspired by the structure of *Modern Robotics: Mechanics, Planning, and Control* by Kevin M. Lynch and Frank C. Park. It does not quote or reproduce textbook prose, exercises, screenshots, or page crops.

## Chapter Question

How does a list of joint motions become a single end-effector transform?

This question is the thread for the chapter. The goal is not to memorize a list of formulas; it is to make the geometry inspectable. A robot mechanism is a physical machine, but the way we reason about it is through spaces, maps, constraints, tangents, and dual forces. The notebook keeps those objects visible. Every diagram and computation below is designed to answer a local question that a reader can check: what is moving, what is fixed, what map is being applied, what invariant should survive, and where can the model fail?

## Translation Guide

The source chapter or appendix is translated into computational language using these terms: home configuration, space screw axis, body screw axis, product of exponentials, URDF, open chain, frame assignment. The important conversions for this notebook are:

- The home pose is the anchor for every exponential product.
- Space and body formulations are equivalent but compose in opposite directions.
- A robot description is useful when geometry and axes agree.

The central habit is to name the representation and the invariant at the same time. Coordinates are useful only when we know what geometric object they represent. A column of joint angles may describe a point on a torus, a homogeneous transform may describe a rigid frame, and a matrix rank may reveal an instantaneous loss of motion. The notebook therefore pairs each formula with a small experiment: a plot, a residual, an ellipsoid, a path, a graph, or a constraint surface.

## Route Through the Notebook

1. Establish the objects and conventions needed for forward kinematics.
2. Build a concept dependency map so definitions have visible structure.
3. Generate the main visual lab: product of exponentials for open chains.
4. Run a compact worked example that exposes an invariant.
5. Try an applied lab that changes a parameter and asks what should remain true.
6. Finish with sanity checks that assert artifact existence, image variation, and numerical margins.

This is a standalone course notebook. The PDF can be useful for historical context and exercises, but the lesson here is complete without it. Definitions are restated in fresh language, examples are computed from scratch, and visuals are generated by course-local code. When the notebook mentions a source span, it is a navigation reference rather than a dependency for comprehension.

## Visualization Storyboard

| Concept | Representation | Artifact | Inspection target |
| --- | --- | --- | --- |
| concept dependency map | directed graph | `artifacts/chapter-04-forward-kinematics/figures/concept-dependency-map.png` | which definitions feed the chapter's computation |
| product of exponentials for open chains | planar arm and end-effector velocity ellipsoid | `artifacts/chapter-04-forward-kinematics/figures/forward-kinematics-lab.png` | how screw axes compose into an end-effector pose |
| numeric invariant check | residual bar chart and JSON summary | `artifacts/chapter-04-forward-kinematics/figures/forward-kinematics-checks.png` | small residuals, positive margins, or rank changes |

The first visual is a dependency map. It is intentionally simple: before computing anything, the reader should see which concepts support which later moves. The second visual is the main lab for this chapter. It turns the chapter's core geometry into something that can be inspected rather than imagined. The third visual is a numeric check: a residual, rank, eigenvalue, path length, or comparable margin that can be tested after the figure is built.

## Working Principles

The most reliable way to learn robotics geometry is to move between three views. The first view is symbolic: equations name maps and constraints. The second view is numerical: a small instance exposes scale, rank, conditioning, and residuals. The third view is visual: geometry becomes a shape, path, ellipsoid, cone, graph, or surface. This notebook keeps all three views close together. If a symbolic statement is correct, the numerical experiment should produce the expected small residual or positive margin. If a visual is meaningful, it should make a specific invariant or failure mode easier to see.

For forward kinematics, the relevant failure modes are not side details; they are part of the concept. A screw axis written in the wrong frame, an end-effector home pose placed at the wrong transform, a multiplication order reversal, or a degree-radian mismatch can all produce a pose that looks plausible while describing the wrong machine. Singularities also belong here: the forward map still returns a transform, but nearby joint changes no longer span all the instantaneous task-space directions the mechanism usually offers. A robust model names those edges, draws them, and then tests a small case so the reader can recognize the issue in later code.

The product-of-exponentials formula is the chapter's organizing map. In the space form, each joint motion is applied as `exp([S_i] theta_i)` before the home transform `M`; in the body form, the same physical pose is described by exponentials after `M` using body-frame screw axes. The two forms are not competing recipes. They are the same geometry viewed from different frames, so the useful questions are: which frame owns this axis, which side of `M` does the exponential act on, and what invariant should survive when the description is changed?

## Applied Lab Preview

Compute a three-link arm pose and compare geometric intuition with matrix products.

In the lab, vary one joint angle at a time and predict how the endpoint and velocity ellipsoid should change before running the code. When the links nearly align, the arm may still have a perfectly good pose, but the Jacobian image becomes thin: one task-space direction is easy to create and the transverse direction is nearly lost. This prediction step is small, but it is what turns a figure into a mathematical instrument.

## Pitfalls To Watch

- Changing joint order changes the pose.
- A link-frame drawing is not a substitute for screw-axis data.

These pitfalls are deliberately operational. If a computation becomes confusing, check frame labels, units, rank, and the distinction between a physical object and its coordinates. Many robotics errors are not arithmetic mistakes; they are mismatches between a representation and the geometry it claims to encode.

## Takeaways

- PoE gives a coordinate-light way to compose joint motion.
- The same model supports visualization, simulation, and later Jacobians.

By the end of this notebook, the reader should be able to explain the chapter's main object, build a small computed example, interpret the generated visual artifact, and state at least one numerical sanity check that would catch an implementation mistake.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

HERE = Path.cwd().resolve()
BOOK_ROOT = next(
    parent for parent in [HERE, *HERE.parents]
    if (parent / "AGENTS.md").exists() and (parent / "Mordern Robotics.pdf").exists()
)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json
from utils.validation import assert_artifact, image_stats

print(f"BOOK_ROOT={BOOK_ROOT}")

In [ ]:
CHAPTER = {
  "number": 4,
  "slug": "chapter-04-forward-kinematics",
  "title": "Forward Kinematics",
  "notebook": "04-forward-kinematics.ipynb",
  "printed_start": 137,
  "printed_end": 170,
  "pdf_start": 155,
  "pdf_end": 188,
  "part_slug": "part-02-manipulator-kinematics",
  "part_title": "Manipulator Kinematics",
  "theme": "kinematics",
  "visual_focus": "product of exponentials for open chains",
  "visual_kind": "planar arm and end-effector velocity ellipsoid",
  "artifact_stem": "forward-kinematics",
  "inspection_target": "how screw axes compose into an end-effector pose",
  "question": "How does a list of joint motions become a single end-effector transform?",
  "terms": [
    "home configuration",
    "space screw axis",
    "body screw axis",
    "product of exponentials",
    "URDF",
    "open chain",
    "frame assignment"
  ],
  "translation": [
    "The home pose is the anchor for every exponential product.",
    "Space and body formulations are equivalent but compose in opposite directions.",
    "A robot description is useful when geometry and axes agree."
  ],
  "lab": "Compute a three-link arm pose and compare geometric intuition with matrix products.",
  "pitfalls": [
    "Changing joint order changes the pose.",
    "A link-frame drawing is not a substitute for screw-axis data."
  ],
  "takeaways": [
    "PoE gives a coordinate-light way to compose joint motion.",
    "The same model supports visualization, simulation, and later Jacobians."
  ]
}

from utils.visuals import build_storyboard
storyboard = build_storyboard(CHAPTER)
storyboard

In [ ]:
from utils.visuals import build_chapter_visuals

outputs = build_chapter_visuals(CHAPTER)
for artifact in outputs['figures']:
    display_artifact(artifact, width=760)
outputs['metrics']

## Worked Example

The worked example below is intentionally compact and reusable. It checks a planar arm Jacobian, a rotation exponential/logarithm round trip, and the twist/wrench power pairing under a frame transform. These are small tests, but they exercise the same representation discipline used throughout the course.

In [ ]:
from utils.kinematics import planar_arm_points, planar_jacobian, manipulability_measure
from utils.lie import adjoint, se3_exp, se3_log, so3_exp, so3_log, transform_from, wrench_power

lengths = np.array([1.0, 0.75, 0.45])
theta = np.array([0.45, -0.8, 0.6])
points = planar_arm_points(lengths, theta)
J = planar_jacobian(lengths, theta)
R = so3_exp(np.array([0.2, -0.1, 0.35]))
round_trip = np.linalg.norm(so3_log(R) - np.array([0.2, -0.1, 0.35]))
T = transform_from(R, np.array([0.25, -0.1, 0.4]))
twist = np.array([0.1, -0.2, 0.3, 0.4, 0.2, -0.1])
wrench = np.array([0.3, 0.1, -0.2, 1.0, -0.4, 0.2])
power_before = wrench_power(twist, wrench)
power_after = wrench_power(adjoint(T) @ twist, np.linalg.inv(adjoint(T)).T @ wrench)
worked_example = {
    "endpoint": points[-1].round(4).tolist(),
    "jacobian_rank": int(np.linalg.matrix_rank(J)),
    "manipulability": float(manipulability_measure(J)),
    "rotation_log_round_trip": float(round_trip),
    "power_pairing_error": float(abs(power_before - power_after)),
}
worked_example

## Applied Lab

The applied lab changes behavior based on the chapter theme. It produces a small numerical summary that should agree with the visual lab: ranks should be plausible, residuals should be small, paths should have nonzero length, and positive-definite quantities should stay positive.

In [ ]:
sweep = np.linspace(-np.pi, np.pi, 90)
values = np.array([
    manipulability_measure(planar_jacobian(np.array([1.0, 0.8]), np.array([0.25, q])))
    for q in sweep
])
worst = int(np.argmin(values))
best = int(np.argmax(values))
lab_summary = {
    "theme": CHAPTER["theme"],
    "max_manipulability": float(values[best]),
    "max_at_joint_2": float(sweep[best]),
    "min_manipulability": float(values[worst]),
    "min_at_joint_2": float(sweep[worst]),
    "near_singular_ratio": float(values[worst] / values[best]),
}

lab_summary

## Sanity Checks

The final cell asserts that the generated artifacts exist, are large enough to be useful, and have nontrivial pixel variation. It also stores a JSON summary under the chapter's artifact subtree so the notebook leaves a machine-checkable trace.

In [ ]:
artifact_stats = {}
for artifact in outputs['figures']:
    resolved = assert_artifact(artifact)
    stats = image_stats(resolved)
    assert stats['pixel_std'] > 2.0, f'low variation image: {resolved}'
    artifact_stats[resolved.name] = stats
assert_artifact(outputs['checks'], min_size=100)
sanity = {'chapter': CHAPTER['slug'], 'metrics': outputs['metrics'], 'worked_example': worked_example, 'lab_summary': lab_summary, 'artifact_stats': artifact_stats}
sanity_path = save_json(sanity, CHAPTER['slug'], 'checks', 'notebook-sanity.json')
display_artifact(sanity_path)
sanity